# Migration Phase 0: Setup Check

Verifies the three new data sources are reachable before we build loaders against them:

1. `brainlink` package + DB (behavioral/demographic data)
2. Tabular neuroimaging derivatives root (anatomical + diffusion)
3. `regional-stacker` package (multivariate regional BAG modeling)

See `data_pipeline_migration_plan.md` for the full plan and resolved decisions.

## 1. Imports

Both new dependencies were added to `pyproject.toml` as local path deps:

```toml
"brainlink @ file:///home/galkepler/Projects/brainlink",
"regional-stacker @ file:///home/galkepler/Projects/regional-stacker",
```

(Required adding `[tool.hatch.metadata] allow-direct-references = true`, since hatchling rejects direct path/URL references by default.)

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv(Path.cwd().parent / ".env")

from brainlink import BrainLinkDB
from regional_stacker import RegionalStackingRegressor, wide_to_stacker_input

print("brainlink + regional-stacker import OK")

brainlink + regional-stacker import OK


## 2. Behavioral data (brainlink)

`BRAINLINK_DB_PATH` in `.env` points at the brainlink SQLite DB. `db.info()` prints a summary of
what's in it (participants, sessions, demographics, questionnaires, imaging paths).

In [2]:
db_path = os.environ["BRAINLINK_DB_PATH"]
assert Path(db_path).exists(), f"brainlink DB not found at {db_path}"

db = BrainLinkDB(db_path)
try:
    db.info()
except Exception as e:
    print(f"db.info() FAILED: {e!r}")

db.info() FAILED: OperationalError('(sqlite3.OperationalError) no such column: session.uid')


### ⚠️ BLOCKER: brainlink DB is empty + ORM/DB schema mismatch

`db.info()` (and any `db.query()`/`db.resolve()` call) currently fails with:

```
OperationalError: (sqlite3.OperationalError) no such column: session.uid
```

**Schema mismatch**: `brainlink/models.py` (current source, 2026-05-13) defines
`participant.uid` / `session.uid` (FK to `participant.uid`), but the **actual
`brainlink.db`** (last modified 2026-06-11, i.e. newer than the models) still has the old
schema with `participant.participant_id` / `session.participant_id`.

**Empty DB**: separately, every table in `brainlink.db` currently has **0 rows** — `participant`,
`session`, `demographics`, `questionnaire_response`, `imaging_path`, etc. are all empty
(checked via raw `PRAGMA`/`COUNT(*)`). No data has been ingested yet.

So the installed `brainlink` package's ORM layer does not match `BRAINLINK_DB_PATH`'s schema,
*and* that DB has no rows regardless. This blocks **Phase 1** (`BehavioralLoader`) until
resolved on the brainlink side — either:
- regenerate/ingest `brainlink.db` from a schema matching current `models.py`
  (`brainlink init` + `ingest`), or
- `models.py` needs a migration step / column-rename for existing DBs, plus an actual
  ingest run.

The cross-check below (§4) falls back to raw `sqlite3` against the actual
`participant_id`-based schema; with 0 rows it returns an empty session list (handled
gracefully), so we can still validate the tabular-derivatives join logic once data exists.

## 3. Tabular neuroimaging derivatives

`TABULAR_DERIVATIVES_ROOT` points at `/mnt/62/Processed_Data/derivatives/tabular`, one
`sub-<UID>/` directory per participant. Each subject has a subject-level `anat/`, plus
per-session `ses-<id>/`, `ses-<id>.cross/` directories with `anat/` and `dwi/` subfolders.

Default config (see `data_pipeline_migration_plan.md`):
- `ATLAS_NAME=Schaefer2018N400n7Tian2020S2` (dwi naming)
- `ANAT_ATLASES=Schaefer2018N400n7,Tian2020S2` (anat cortex+subcortex, concatenated)
- `SESSION_VARIANT=cross` (prefer `ses-<id>.cross/` for max session coverage)

In [3]:
tabular_root = Path(os.environ["TABULAR_DERIVATIVES_ROOT"])
assert tabular_root.exists(), f"tabular derivatives root not found at {tabular_root}"

subject_dirs = sorted(tabular_root.glob("sub-*"))
print(f"{len(subject_dirs)} subject directories found")
print("example:", subject_dirs[0].name)

# spot-check: does the example subject have the expected anat atlases?
example = subject_dirs[0]
for atlas in os.environ["ANAT_ATLASES"].split(","):
    p = example / "anat" / f"atlas-{atlas}"
    print(atlas, "->", "OK" if p.exists() else "MISSING", p)

2828 subject directories found
example: sub-S000091
Schaefer2018N400n7 -> OK /mnt/62/Processed_Data/derivatives/tabular/sub-S000091/anat/atlas-Schaefer2018N400n7
Tian2020S2 -> OK /mnt/62/Processed_Data/derivatives/tabular/sub-S000091/anat/atlas-Tian2020S2


In [4]:
import sqlite3
import pandas as pd

# db.query() is broken (see blocker above) - fall back to raw sqlite against the
# actual participant_id-based schema for this sanity check.
con = sqlite3.connect(db_path)
sessions = pd.read_sql(
    "SELECT session_id, participant_id AS uid, subject_code FROM session "
    "WHERE participant_id IS NOT NULL AND subject_code IS NOT NULL",
    con,
)
con.close()
print(f"{len(sessions)} sessions with uid + subject_code")

sample = sessions.sample(min(10, len(sessions)), random_state=42)
for _, row in sample.iterrows():
    sub_dir = tabular_root / f"sub-{row['uid']}"
    cross_dir = sub_dir / f"ses-{row['session_id']}.cross"
    plain_dir = sub_dir / f"ses-{row['session_id']}"
    print(
        row["uid"], row["session_id"],
        "cross:", cross_dir.exists(),
        "plain:", plain_dir.exists(),
    )

0 sessions with uid + subject_code


## 5. Note on unrelated dependency drift

`uv sync --all-extras` removed several packages that were present in the venv but never
declared in `pyproject.toml` (e.g. `umap-learn`, `streamlit`, `pyvista`, `surfplot`, `vtk`,
`subcortex-visualization`). These are used by `src/neuroalign/visualization/brain.py` and
`app/main.py` (separate WIP, unrelated to this migration). This drift pre-dates this branch
(confirmed against the prior `uv.lock`) — flagged here, not fixed as part of this migration.